# A memory that knows when it might be wrong
### Confidence-gated memory guidance for invoice OCR: a guided demo

An OCR model reads every invoice as if it had never seen one before. Vendors print the same layout over and over,
so a **memory of where each field sits** should help. In v2 of this project it did, for vendors the memory knew.
For vendors it *didn't* know, the memory handed over a stranger's layout and box accuracy fell by 0.13 IoU.

This demo walks through the answer: a **memory graph** in Neo4j, and a **gate** that shows the model a remembered
position only when it can say how likely that position is to be right.

| Part | Question |
|---|---|
| **I. The memory graph** | What is remembered, and how do raw observations become rules? |
| **II. Confidence gating** | Which remembered layout is this page, and how far can each rule be trusted? |
| **III. Evaluation, no LLM** | What can the gate do on generated test pages: look-alike vendors, missing vendor names, unseen vendors, redesigned layouts? |
| **IV. Does it help the LLM?** | A short summary of the Gemini runs in `gating.ipynb` |

Everything up to Part IV runs without any LLM call. The code lives in `src/`, and this notebook only tells the
story.

### Setup
Settings live in `src/config.py` (the same values as `gating.ipynb`). Override them here. `data_dir` must hold
`input/` (PDFs) and `output/` (ground-truth JSON). Neo4j runs in its own container:
```
docker run --name graph_gating -d -p 7475:7474 -p 7688:7687 -e NEO4J_AUTH=neo4j/gating-local neo4j:5.26-community
```

In [ ]:
import warnings
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

import helpers as H
from src import viz
from src import evaluation as E
from src.config import Config
from src.data import NEW, Corpus
from src.gate import Gate
from src.memory_graph import ALL_VENDORS, BOX_KEYS, MemoryGraph
from src.posterior import LayoutPosterior
from src.scenarios import ScenarioGenerator, case_table, describe_noise, entity_agreement, find_twins

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

cfg = Config(
    data_dir=Path.cwd().parent / "data",     # <- point at the folder holding input/ and output/
    neo4j_uri="bolt://localhost:7688", neo4j_user="neo4j", neo4j_password="gating-local",
)
REBUILD_GRAPH = True                         # False keeps an already-built graph for the same split

In [ ]:
corpus = Corpus.load(cfg)
graph = MemoryGraph(cfg)
graph.build(corpus, reset=REBUILD_GRAPH)
post = LayoutPosterior(graph, corpus, cfg)
gate = Gate(post, graph)

vendors = corpus.vendor_table()
print(f"{len(corpus.records)} invoices from {len(vendors)} vendors | memory built from {len(corpus.train_ids)} "
      f"| held out {len(corpus.test_ids)}")
corpus.bucket_table()

The split is **per vendor**: two thirds of each vendor's invoices build the memory, and the rest are held out. A
vendor with only one invoice lands entirely on one side. If it lands in the held-out third, it is a vendor the
memory has **never seen**. The training-set size buckets above recur throughout the evaluation.

In [ ]:
# The worked examples, picked by rule so the demo is the same on any machine
EX_VENDOR = vendors[vendors.n_test > 0].sort_values("n_train", ascending=False).vendor_key.iloc[0]
EX_DOC = sorted(d for d in corpus.test_ids if corpus.vendor_of(d) == EX_VENDOR)[0]
EX_PAGE = corpus.records[EX_DOC]["pages"][0]
EX_LAYOUT = graph.cypher("MATCH (:Vendor {key: $v})-[:HAS_LAYOUT]->(p:PageLayout {page_number: 1}) "
                         "RETURN p.id AS id ORDER BY p.doc_count DESC LIMIT 1", v=EX_VENDOR)[0]["id"]
print(f"Running example: {EX_DOC}, page 1, vendor '{EX_VENDOR}' "
      f"({corpus.train_count[EX_VENDOR]} invoices in memory) -> layout {EX_LAYOUT}")

---
## Part I: The memory graph

### 0. What the model has to find
A held-out invoice from the vendor with the most invoices in memory. Each blue box is a ground-truth **value** the
OCR model must read and locate. The values of one vendor tend to sit in the same places on every invoice. That
regularity is what the memory stores.

In [ ]:
viz.page_with_fields(corpus.pdf(EX_DOC), EX_PAGE, title=f"{EX_DOC} · page 1 · ground truth")
plt.show()

### 1. The schema: one path per field
The memory is a tree per vendor. A node is not a document. It is a **path in memory**: "vendor X, page 1, layout
variant 0, section Primary Fields, entity Invoice Amount". Every invoice of that vendor that is merged into the
tree adds one **observation** to the arrays on that path.

```
(:DocumentType)◄─OF_TYPE─(:Vendor)─HAS_LAYOUT─►(:PageLayout)─HAS_SECTION─►(:SectionLayout)─HAS_ENTITY─►(:EntityLayout)
                                                     │                                                ├─NORMALIZED_BY─►(:NormalizationRule)
                                                     └─HAS_TABLE─►(:TableLayout)─HAS_COLUMN─►(:ColumnLayout)─CONSTRAINED_BY─►(:Invariant)
```

Below is one path of every kind from the running example's tree. Shared nodes (`DocumentType`,
`NormalizationRule`, `Invariant`) are linked from every tree.

In [ ]:
display(HTML(viz.legend_html(viz.LABEL_COLORS)))
viz.show_graph(graph.driver, '''
    MATCH (t:DocumentType)<-[o:OF_TYPE]-(v:Vendor {key: $vendor})-[h:HAS_LAYOUT]->(p:PageLayout {id: $lid})
    MATCH (p)-[hs:HAS_SECTION]->(s:SectionLayout {name: 'Primary Fields'})-[he:HAS_ENTITY]->(e:EntityLayout)
    WHERE e.key IN ['Invoice Amount', 'Invoice Date']
    OPTIONAL MATCH (e)-[n:NORMALIZED_BY]->(r:NormalizationRule)
    OPTIONAL MATCH (e)-[c:CONSTRAINED_BY]->(i:Invariant)
    OPTIONAL MATCH (p)-[ht:HAS_TABLE]->(tl:TableLayout)-[hc:HAS_COLUMN]->(cl:ColumnLayout)
    WHERE cl.header IN ['Line Item Amount', 'Item']
    OPTIONAL MATCH (cl)-[cc:CONSTRAINED_BY]->(i)
    RETURN *''', vendor=EX_VENDOR, lid=EX_LAYOUT, layout="hierarchical")

In [ ]:
graph.node_counts()

### 2. Writing memory: ingestion
Each training page **walks its path**: `MERGE` creates whatever part of the path is missing, then the page
**appends** one observation to every entity on it: filled or not, the value's box (page fractions), and the label
printed next to it. Re-running is idempotent, because a document already in `obs_doc_ids` is skipped. Every page
is also merged into a corpus-wide tree, `ALL_VENDORS`.

The running example's page-1 tree is shown below, with entities sized by how often they were filled.

In [ ]:
display(HTML(viz.legend_html({k: viz.LABEL_COLORS[k] for k in ("Vendor", "PageLayout", "SectionLayout", "EntityLayout")})))
viz.show_graph(graph.driver, '''
    MATCH path = (v:Vendor {key: $vendor})-[:HAS_LAYOUT]->(p:PageLayout {id: $lid})-[:HAS_SECTION]->()
                 -[:HAS_ENTITY]->(e:EntityLayout)
    WHERE e.n_filled > 0
    RETURN path''', vendor=EX_VENDOR, lid=EX_LAYOUT,
    size=lambda n: 6 + 2.5 * n.properties.get("n_filled", 0) if n.properties["labels"][0] == "EntityLayout" else 16)

In [ ]:
obs = graph.entity_observations(EX_LAYOUT, "Invoice Amount")
print(f"The 'Invoice Amount' path of {EX_LAYOUT} holds one observation per merged page:")
pd.DataFrame({"doc_id": obs["doc_ids"], "x": obs["x"], "y": obs["y"], "w": obs["w"], "h": obs["h"]}).round(3)

### 3. One vendor, several layouts
A vendor doesn't always print the same layout: a page 2 that only continues the line items looks nothing like
page 1. Pages of one vendor and page number share a **layout variant** when their fill vectors (which fields are
present) have cosine ≥ `structure_similarity` (0.85). Otherwise they open a new variant. Each variant is its own
`PageLayout` with its own rules.

In [ ]:
multi = graph.cypher_df('''
    MATCH (v:Vendor)-[:HAS_LAYOUT]->(p:PageLayout) WHERE v.key <> $all
    WITH v, p.page_number AS page, collect(p) AS ps WHERE size(ps) >= 2
    RETURN v.key AS vendor, page, size(ps) AS variants, [x IN ps | x.doc_count] AS docs
    ORDER BY reduce(s = 0, d IN docs | s + d) DESC''', all=ALL_VENDORS)
display(multi.head(8))
VAR_VENDOR = multi.vendor.iloc[0] if not multi.empty else EX_VENDOR
variant_colors = [viz.ORANGE, viz.VIOLET, viz.MAGENTA, viz.GREEN]
viz.show_graph(graph.driver, '''
    MATCH path = (v:Vendor {key: $vendor})-[:HAS_LAYOUT]->(p:PageLayout)-[:HAS_TABLE]->(t:TableLayout)
    RETURN path''', vendor=VAR_VENDOR,
    color=lambda n: variant_colors[n.properties["variant"] % 4] if n.properties["labels"][0] == "PageLayout"
                    else viz.LABEL_COLORS[n.properties["labels"][0]])

In [ ]:
fv = graph.cypher_df('''MATCH (:Vendor {key: $vendor})-[:HAS_LAYOUT]->(p:PageLayout)
                        RETURN p.id AS layout, p.doc_count AS docs, p.fill_vector AS fill''', vendor=VAR_VENDOR)
features = ["(constant)"] + H.KV_KEYS + list(H.TABLE_TYPES)
fill = pd.DataFrame(fv.fill.tolist(), index=fv.layout, columns=features).T.round(2)
print("Share of pages on which each field is present, per variant (only rows where variants differ):")
fill[fill.max(axis=1) - fill.min(axis=1) > 0.3]

### 4. Consolidation: from observations to a rule with a reliability
For one entity in one layout:

1. **Peak.** The observed box centre with the most other centres within `bbox_tolerance` (0.03 of the page).
   **Support** is that count.
2. **Strength.** The entity is *strong* with ≥ 3 agreeing documents, *weak* with 2, and *single* when the layout
   has one document.
3. **Range.** A square around the peak, half-width `clip(2 × spread, 0.06, 0.15)`. Extracted boxes are noisy
   (Gemini's centres miss by ~0.057), so a rule must be matched against a region, not a point.
4. **Reliability, `p_in_range`.** Every training box is a trial: inside the range or not. With an empirical-Bayes
   Beta prior, $q = \frac{(s-1) + \alpha}{(n-1) + \alpha + \beta}$. The peak box is **not** counted, because the
   range is centred on it, so it can't miss.

The plot shows every observed centre of one entity of the running layout, its peak, its range and its typical box.

In [ ]:
rules_tbl = graph.rules_table()
cand = rules_tbl[(rules_tbl.layout == EX_LAYOUT) & (rules_tbl.n_boxes >= 5)].assign(
    outside=lambda d: d.n_boxes - d.n_in_range).sort_values(["outside", "n_boxes"], ascending=False)
EX_KEY = cand.entity.iloc[0]
o = graph.entity_observations(EX_LAYOUT, EX_KEY)
centres = [(x + w / 2, y + h / 2) for x, y, w, h in zip(o["x"], o["y"], o["w"], o["h"]) if x >= 0]

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 7.5), gridspec_kw={"width_ratios": [1.15, 1]})
viz.page_axes(corpus.pdf(EX_DOC), 1, ax, title=f"'{EX_KEY}' in {EX_LAYOUT}", fade=0.45)
ax.scatter(*zip(*centres), s=28, color=viz.BLUE, edgecolor="white", lw=1, zorder=5, label="observed centres")
ax.scatter(*o["center"], s=90, marker="X", color=viz.ORANGE, edgecolor="white", lw=1, zorder=6, label="peak")
viz.draw_range(ax, o["range"], viz.ORANGE, f"range · q = {o['q']:.2f}")
viz.draw_box(ax, o["median"], viz.INK, None, lw=1.2)
ax.legend(loc="lower left", fontsize=8, frameon=True)

by_strength = rules_tbl.groupby("strength").p_in_range.describe()[["count", "mean", "min", "max"]]
ax2.axis("off")
ax2.set_title("All vendor rules: strength label vs p_in_range", loc="left")
ax2.table(cellText=by_strength.round(2).values, rowLabels=by_strength.index, colLabels=by_strength.columns,
          loc="upper left", bbox=[0.1, 0.62, 0.9, 0.3])
ax2.text(0, 0.45, f"support {o['support']} of {o['n_boxes']} boxes\n"
                  f"{o['n_in_range']} of {o['n_boxes']} inside the range\n"
                  f"strength '{o['strength']}' · p_in_range {o['q']:.3f}\n\n"
                  f"Empirical prior: a one-document rule catches the\nnext document "
                  f"{graph.rule_prior[2]:.0%} of the time -> Beta({graph.rule_prior[0]:.2f}, {graph.rule_prior[1]:.2f})",
         va="top", fontsize=9, color=viz.INK_2)
plt.show()

In [ ]:
display(HTML(viz.legend_html({f"{k} rule": c for k, c in viz.STRENGTH_COLORS.items()})))
viz.show_graph(graph.driver, '''
    MATCH path = (v:Vendor {key: $vendor})-[:HAS_LAYOUT]->(p:PageLayout {id: $lid})-[:HAS_SECTION]->()
                 -[:HAS_ENTITY]->(e:EntityLayout)
    WHERE e.n_filled > 0
    RETURN path''', vendor=EX_VENDOR, lid=EX_LAYOUT,
    caption=lambda n: (f"{n.properties['key']} · q={n.properties['p_in_range']:.2f}"
                       if n.properties["labels"][0] == "EntityLayout" and n.properties.get("p_in_range") is not None
                       else viz.default_caption(n)),
    color=lambda n: viz.STRENGTH_COLORS.get(n.properties.get("strength"), viz.LABEL_COLORS[n.properties["labels"][0]]))

### 5. `ALL_VENDORS`: what is true of invoices in general
Every training page is also merged into one corpus-wide tree. It holds no useful positions, because the same field
is printed all over the page across vendors. It does hold **how often** each field is present and **which labels**
are common. It is the memory's model of "an invoice from someone I don't know", the **NEW** class of Part II.

In [ ]:
corpus_obs = graph.entity_observations(f"{ALL_VENDORS}|p1|v0", EX_KEY)
all_centres = [(x + w / 2, y + h / 2) for x, y, w, h in zip(corpus_obs["x"], corpus_obs["y"], corpus_obs["w"],
                                                               corpus_obs["h"]) if x >= 0]
ax = viz.page_axes(corpus.pdf(EX_DOC), 1, title=f"'{EX_KEY}': every vendor (grey) vs '{EX_VENDOR}' (blue)", fade=0.6)
ax.scatter(*zip(*all_centres), s=16, color=viz.MUTED, alpha=0.7, zorder=4)
ax.scatter(*zip(*centres), s=26, color=viz.BLUE, edgecolor="white", lw=1, zorder=5)
viz.draw_range(ax, o["range"], viz.BLUE, f"{EX_VENDOR} range")
plt.show()
print("Corpus-wide labels:", {k: v for k, v in post.corpus_layouts[1]["labels"].items() if v and k in BOX_KEYS})

---
## Part II: Confidence gating

At test time, the memory sees a page only through an **unguided extraction**: the vendor name it printed, which
fields are filled, and roughly where the boxes are. In the real pipeline this is Gemini's call 0. In this demo it
is a **generated test page**: the held-out ground truth passed through a noise channel tuned to Gemini's measured
errors (box centres off by ~0.057, 5% of fields missed, 2% invented, 10% of names misread). Part III explains
the generator.

In [ ]:
gen = ScenarioGenerator(corpus, graph, cfg)
baseline_cases = gen.baseline()
unseen_cases = [c for c in gen.unseen_and_drift() if c.scenario == "unseen vendor"]

EX_CASE = next(c for c in baseline_cases if c.doc_id == EX_DOC and c.page == 1 and c.scenario == "noisy")
ung_rows = E.evaluate_ungated(graph, post, unseen_cases)
worst = ung_rows.groupby("case_id").hit.agg(lambda h: (~h).sum()).idxmax()
UNSEEN_CASE = next(c for c in unseen_cases if c.case_id == worst)
print(f"Running example (known vendor): {EX_CASE.case_id}, name read as '{EX_CASE.vendor_name}'")
print(f"Contrast (vendor never seen):   {UNSEEN_CASE.case_id}, name read as '{UNSEEN_CASE.vendor_name}'")

### 6. Naive recall: the ungated guide (pass 1)
v2's guide takes the nearest layout by cosine of **structure vectors** (which fields, plus roughly where:
a soft 4×4 grid per field). It hands over **every** rule of that layout. For a known vendor it searches that
vendor's layouts. For an unseen vendor it searches everyone's, and it always finds a "nearest" layout, however
poor the match.

The contrast page comes from a vendor the memory has never seen. Its nearest neighbours are shown below. Edge
captions are cosine similarities, and the guide hands over the top two.

In [ ]:
c = UNSEEN_CASE
structure = H.structure_vector(c.evidence, cfg.position_grid, cfg.position_weight)
near = graph.nearest_layouts(structure, c.page, k=6)
nodes = [{"id": "page", "caption": f"test page: '{c.vendor_name}' (unseen)", "color": viz.INK_2, "size": 22}]
edges = []
for i, n in enumerate(near):
    nodes.append({"id": n["layout_id"], "caption": f"{n['vendor']} ({n['docs']} docs)",
                  "color": viz.ORANGE if i < cfg.guide_neighbours else viz.MUTED, "size": 10 + 4 * n["docs"]})
    edges.append({"source": "page", "target": n["layout_id"], "caption": f"cos {n['structure_similarity']:.2f}",
                  "width": 3 if i < cfg.guide_neighbours else 1})
display(HTML(viz.legend_html({"handed to the model": viz.ORANGE, "not used": viz.MUTED})))
viz.show_virtual(nodes, edges)

In [ ]:
guide, shown = E.ungated_shown(graph, post, c)
gt_page = c.gt
fig, ax = plt.subplots(figsize=(7, 9))
viz.page_axes(corpus.pdf(c.doc_id), c.page, ax, title="Ungated guide on an unseen vendor: blue = truth, orange = memory")
for key, f in gt_page["fields"].items():
    if f["bbox"]:
        viz.draw_box(ax, f["bbox"], viz.BLUE)
for key, r in shown.items():
    viz.draw_range(ax, r["range"], viz.ORANGE, key)
plt.show()
hits = [H.in_range(H.bbox_center(gt_page["fields"][k]["bbox"]), r["range"]) for k, r in shown.items()
        if gt_page["fields"].get(k, {}).get("usable")]
print(f"The ungated guide shows {len(shown)} positions; {sum(hits)} of the {len(hits)} checkable ones are right.")

### 7. Which remembered layout is this page? The posterior
Instead of "nearest", ask **how probable** each remembered layout is, given the evidence $e$, including the
possibility that it is none of them (**NEW**):

$$P(L \mid e) \;\propto\; \underbrace{P(L)}_{\text{prior}}\;\underbrace{P(\hat k \mid L)}_{\text{name}}\;
\underbrace{P(x \mid L)^{1/T}}_{\text{which fields}}\;\underbrace{p(c \mid x, L)}_{\text{where the boxes are}}$$

* **prior:** NEW gets the Good–Turing mass $N_1/N$ (the share of training invoices whose vendor appears once).
  Known layouts share the rest by size.
* **name:** a noisy channel. The right key with probability 0.9, a variant spelling or another key otherwise. A
  **missing** name is no evidence.
* **fields:** smoothed fill rates, seen through the extraction's own miss and false-fill rates.
* **boxes:** each box centre inside a layout's range scores $q/|R|$, and outside it $(1-q)/(1-|R|)$.

The bars show each term relative to the best candidate. A long negative bar is the term that ruled a candidate out.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 9))
for ax, c, title in ((axes[0], EX_CASE, "Known vendor"), (axes[1], UNSEEN_CASE, "Vendor never seen")):
    viz.term_bars(post.explain(c.evidence, c.vendor_name, top=5), ax=ax,
                  title=f"{title}: '{c.vendor_name}' (truth: {c.truth})")
plt.tight_layout()
plt.show()
print(f"P(NEW) prior = {post.p_new:.2f}: most training vendors appear only once, so 'someone new' is the base case.")

In [ ]:
g = gate.page_rules(EX_CASE.evidence, EX_CASE.vendor_name)
nodes = [{"id": "page", "caption": f"test page '{EX_CASE.vendor_name}'", "color": viz.INK_2, "size": 20}]
edges = []
for lid, p in g["top_layouts"].items():
    vendor = "NEW (not in memory)" if lid == NEW else post.vendor_layouts[lid]["vendor"]
    nodes.append({"id": lid, "caption": f"{vendor} · P={p:.2f}", "size": 8 + 40 * p,
                  "color": viz.MUTED if lid == NEW else viz.ORANGE})
    edges.append({"source": "page", "target": lid, "caption": f"{p:.2f}", "width": 1 + 6 * p})
viz.show_virtual(nodes, edges, height="380px")

### 8. From layout belief to a confidence for each rule
The gate shows the rules of the most probable layout $L^*$. How far to trust *its* rule for field $k$? By the law
of total probability over which layout the page really is:

$$\text{conf}_k = P(c_k \in R_{L^*,k} \mid e) = \sum_L P(L \mid e)\; P(c_k \in R_{L^*,k} \mid L)$$

| if the page really is… | chance the value is in $L^*$'s range |
|---|---|
| $L^*$ | $q$, the rule's own reliability |
| another layout with a range for $k$ | how much of its probability mass falls in $R^*$ |
| another layout without one, or NEW | the area of $R^*$ (~0.01–0.09) |

So a rule stays trusted when the runner-up layout **puts the value in the same place**. It is not just
$P(L^* \mid e)$. It is not just $q$ either, because a reliable rule of the wrong layout is still wrong.

The example is the generated page on which the posterior is **most torn** between a vendor and its structural twin
(two vendors whose invoices look alike, see E2). The two tables break down the confidence of one field the twins
print in the same place, and one they print in different places.

In [ ]:
twins = find_twins(graph, cfg.twin_min_similarity, cfg.twin_min_shared_rules)
twin_cases = gen.twins(twins)


def ambiguity(c):   # how evenly the posterior splits between the page's vendor and its twin
    vp = gate.page_rules(c.evidence, c.vendor_name)["vendor_posterior"]
    return min(vp.get(c.true_vendor, 0.0), vp.get(c.meta["twin"], 0.0))


TWIN_CASE = max((c for c in twin_cases if not all(c.meta["agree"].values())), key=ambiguity)
TWIN_A, TWIN_B = TWIN_CASE.true_vendor, TWIN_CASE.meta["twin"]
LAYOUT_A, LAYOUT_B = TWIN_CASE.meta["own_layout"], TWIN_CASE.meta["twin_layout"]
print(f"Twins: '{TWIN_A}' and '{TWIN_B}' (structure cosine {TWIN_CASE.meta['twin_similarity']:.2f}). "
      f"Test page of '{TWIN_A}', scenario '{TWIN_CASE.scenario}', name read as '{TWIN_CASE.vendor_name}'.")
g = gate.page_rules(TWIN_CASE.evidence, TWIN_CASE.vendor_name)
print(f"P({TWIN_A}) = {g['vendor_posterior'].get(TWIN_A, 0):.2f}, P({TWIN_B}) = {g['vendor_posterior'].get(TWIN_B, 0):.2f}"
      f" -> MAP layout {g['chosen']}")
agree = TWIN_CASE.meta["agree"]
for key in [k for k, v in agree.items() if v][:1] + [k for k, v in agree.items() if not v][:1]:
    print(f"\n'{key}' (twins {'agree' if agree[key] else 'disagree'} on where it is):")
    display(gate.confidence_breakdown(TWIN_CASE.evidence, TWIN_CASE.vendor_name, key, top=4).round(3))

In [ ]:
la, lb = post.vendor_layouts[LAYOUT_A], post.vendor_layouts[LAYOUT_B]
fig, ax = plt.subplots(figsize=(7, 9))
viz.page_axes(corpus.pdf(TWIN_CASE.doc_id), TWIN_CASE.page, ax, fade=0.4,
              title=f"Twin ranges: {TWIN_A} (blue) vs {TWIN_B} (orange)")
for key, same in agree.items():
    viz.draw_range(ax, la["rules"][key]["range"], viz.BLUE, key if not same else None, ls="-")
    viz.draw_range(ax, lb["rules"][key]["range"], viz.ORANGE, None)
plt.show()

### 9. The gate
The confidence becomes a decision:

| confidence | what the model is shown |
|---|---|
| ≥ 0.7 (`gate_box`) | the rule's typical box and its confidence |
| 0.4 – 0.7 (`gate_label`) | label hint only, "position uncertain" |
| < 0.4 | nothing: the rule is **withheld** |

Below are the running example's rules and the MAP layout's entities in the graph, colored by the gate's decision.
The last cell shows the two prompts side by side.

In [ ]:
g = gate.page_rules(EX_CASE.evidence, EX_CASE.vendor_name)
decided = {r["name"]: r["gate"] for r in g["rules"]}
display(pd.DataFrame(g["rules"])[["name", "strength", "q", "p_filled", "confidence", "gate"]]
        .dropna(subset=["confidence"]).round(3))
display(HTML(viz.legend_html(viz.GATE_COLORS)))
viz.show_graph(graph.driver, '''
    MATCH path = (p:PageLayout {id: $lid})-[:HAS_SECTION]->()-[:HAS_ENTITY]->(e:EntityLayout)
    WHERE e.bbox_range IS NOT NULL
    RETURN path''', lid=g["chosen"],
    color=lambda n: viz.GATE_COLORS.get(decided.get(n.properties.get("key")), viz.LABEL_COLORS[n.properties["labels"][0]]))

In [ ]:
ungated_text = graph.rules_to_text(graph.memory_guide(EX_CASE.vendor_name, EX_CASE.evidence))
gated_text = gate.rules_to_text(g)
unseen_text = gate.rules_to_text(gate.page_rules(UNSEEN_CASE.evidence, UNSEEN_CASE.vendor_name))
style = "font:11px ui-monospace,monospace;white-space:pre-wrap;padding:8px;border:1px solid #e1e0d9;border-radius:6px"
display(HTML(f'''<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">
  <div><b>Ungated prompt (pass 1)</b><div style="{style}">{ungated_text}</div></div>
  <div><b>Gated prompt (pass 2)</b><div style="{style}">{gated_text}</div>
       <b>Gated prompt, unseen vendor</b><div style="{style}">{unseen_text}</div></div></div>'''))

---
## Part III: Evaluation without an LLM

The gate makes a claim per rule: *"this value is in this range with probability conf"*. That claim can be checked
with ground truth alone: a rule **hits** when the true value centre falls inside its range. So the gate can be
evaluated on **generated test pages**, with no Gemini calls.

**The generator** (`src/scenarios.py`) builds controlled variants of the held-out pages, each with a known right
answer:

| study | generated pages | right answer |
|---|---|---|
| **E1 baseline** | every held-out page, clean, and through the noise channel | its vendor, or NEW if unseen |
| **E2 look-alike vendors** | pages of vendors with a *structural twin*, shown with their own name, no name, or the twin's name | its own vendor |
| **E3 no vendor** | every held-out page with the vendor name removed | its vendor, or NEW |
| **E4 unseen & drift** | unseen vendors; known layouts under a never-seen name; known names on a **redesigned** layout (all boxes moved) | NEW for unseen and redesigned |

Every study is also scored against the **ungated guide** on the same pages. The metric that matters most to an LLM
is **wrong positions per page**: confident boxes that point at the wrong spot.

In [ ]:
cases = gen.all_studies()
pages, rules = E.evaluate(gate, cases)
ungated = E.evaluate_ungated(graph, post, cases)
summary = E.study_summary(pages, rules)
effect = E.memory_effect(pages, rules, ungated)
display(case_table(cases))
print("What the noise channel did (target: centre miss 0.057, 5% dropped, 2% invented):")
describe_noise(cases)

### E1. Baseline: is the confidence honest?
On ordinary held-out pages: how often does the posterior find the right group, how much of the memory passes the
gate, and do rules with confidence 0.9 hit about 90% of the time? The **reliability curve** should follow the
diagonal. On the **risk–coverage curve**, accuracy should rise as the threshold rises. If it's flat, the
confidence isn't ranking good rules above bad ones.

In [ ]:
display(summary.loc["E1 baseline"])
display(effect.loc["E1 baseline"])
groups = {"E1 noisy pages": rules[(rules.study == "E1 baseline") & (rules.scenario == "noisy")],
          "all generated pages (E1–E4)": rules}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.4))
viz.reliability_plot(groups, ax=a1)
viz.coverage_plot(groups, ax=a2, gate_box=cfg.gate_box)
plt.tight_layout(); plt.show()
for name, df in groups.items():
    print(name, {k: round(v, 3) for k, v in E.calibration_scores(df).items()})
display(E.reliability(rules.confidence, rules.hit))

**What to notice.** On the same pages, ungated memory points at the wrong spot about 2 times per page. The gate
keeps about two thirds of those positions and gets ~95% of them right, so wrong positions fall to ~0.2 per page.
Most of the confidence mass sits above 0.9, and there the hit rate matches. The curve is noisier in the middle
bins, which hold few rules.

### E2. Similar structure, different vendors
Some vendors print near-identical invoices: the same fields, in the same places (structure cosine ≥ 0.9, at least
3 located fields in common). These **twins** are the hardest case for identification, and the clearest case for
why rule confidence is not the same as layout confidence.

Each test page of a twinned vendor is shown three ways: with its own name, with **no name**, and with the **twin's
name** (the extraction misread the vendor).

In [ ]:
display(twins[["vendor_a", "vendor_b", "page", "sim", "shared_rules", "agreeing_rules"]].head(10).round(3))
agree_color = {True: viz.AQUA, False: viz.RED}
side_color = {TWIN_A: viz.BLUE, TWIN_B: viz.ORANGE, LAYOUT_A: viz.BLUE, LAYOUT_B: viz.ORANGE}


def twin_color(n):
    label, props = n.properties["labels"][0], n.properties
    if label == "Vendor":
        return side_color[props["key"]]
    if label == "PageLayout":
        return side_color[props["id"]]
    if label == "EntityLayout":
        return agree_color[agree[props["key"]]]
    return viz.LABEL_COLORS[label]


display(HTML(viz.legend_html({TWIN_A: viz.BLUE, TWIN_B: viz.ORANGE, "twins agree on position": viz.AQUA,
                              "twins disagree": viz.RED})))
viz.show_graph(graph.driver, '''
    MATCH path = (v:Vendor)-[:HAS_LAYOUT]->(p:PageLayout)-[:HAS_SECTION]->()-[:HAS_ENTITY]->(e:EntityLayout)
    WHERE p.id IN [$a, $b] AND e.key IN $keys
    RETURN path''', a=LAYOUT_A, b=LAYOUT_B, keys=list(agree), color=twin_color,
    caption=lambda n: n.properties.get("key") or n.properties.get("name") or n.properties.get("id"))

In [ ]:
twin_cases = [c for c in cases if c.study == "E2 twins"]
twins["disagreeing_rules"] = twins.shared_rules - twins.agreeing_rules
print("Which evidence tells twins apart? (top-1 = the right vendor has the highest posterior)")
display(E.identification(post, twin_cases)["top1_acc"].unstack("evidence"))
display(summary.loc["E2 twins"][["pages", "map_acc", "p_true", "share_box", "hit_box", "brier"]])
print("Rules split by whether the twins' ranges overlap:")
E.twin_agreement(rules)

**What to notice.**
* **Names separate twins and boxes come second.** Without a name, positions (`boxes`) carry most of the
  identification. With the twin's name, the name channel actively misleads and top-1 drops to about a coin flip.
* **Yet box precision barely moves.** When the MAP layout is the twin's, the rules it shows are mostly still
  right, because the twins print those fields in the same place. The confidence formula credits that agreement:
  rules where twins agree keep high confidence *and* high hit rates.
* **The weak spot:** where twins *disagree*, a misread name still yields ~0.87 confidence against a lower hit
  rate. This is the gate's honest limit. When the evidence confidently points at the wrong twin, the
  total-probability sum has nothing to correct with.

### E3. Documents with no vendor
In this corpus every ground-truth invoice names its vendor. But a missing or illegible name is common in
extraction. E3 removes the name from **every** held-out page, so the posterior has to place the page from its
structure alone. For pages of unseen vendors, the right answer is still NEW.

In [ ]:
no_vendor = [c for c in cases if c.study == "E3 no vendor"]
ident = E.identification(post, no_vendor)
display(ident["top1_acc"].unstack("evidence"))
display(pd.concat({"with name (E1)": summary.loc["E1 baseline"], "no name (E3)": summary.loc["E3 no vendor"]})
        [["pages", "map_acc", "p_new", "share_box", "share_withheld", "hit_box", "brier"]])
display(pd.concat({"with name (E1)": effect.loc["E1 baseline"], "no name (E3)": effect.loc["E3 no vendor"]}))
by_bucket = pages[pages.study == "E3 no vendor"].assign(bucket=lambda d: d.true_vendor.map(corpus.bucket))
by_bucket.groupby(["scenario", "bucket"]).agg(pages=("page", "size"), map_acc=("correct", "mean"),
                                              p_new=("p_new", "mean")).round(3)

**What to notice.**
* Without a name, identification rests on structure, and **positions do most of the work** (boxes alone beat
  fields alone). MAP accuracy falls from ~0.9 to ~0.74.
* The cost is highest for **thin trees** (2–3 invoices in memory). Clean pages of unseen vendors are also more
  often claimed by a known layout than noisy ones. Exact positions can make a stranger's page resemble a known
  layout, while noise breaks the resemblance and sends the page to NEW.
* **The gate absorbs most of it.** Box precision stays ~0.94 and wrong positions per page barely move, because a
  wrong-but-similar layout usually shares the positions it is confident about.

### E4. Unseen vendors and layout drift
This is where the ungated memory did damage in v2.
* **Unseen vendor:** real held-out vendors with no invoices in memory.
* **Known layout, new name:** a known vendor's page under a name the memory has never seen (a rebrand, or a
  badly misread name). Should structure override the name?
* **Known name, redesigned layout:** the vendor's own name, but every box moved down a quarter page. The name
  says "trust me" and the boxes say otherwise.

In [ ]:
display(summary.loc["E4 unseen & drift"][["pages", "map_acc", "p_true", "p_new", "rules", "share_box", "hit_box"]])
display(effect.loc["E4 unseen & drift"])

In [ ]:
redesign = next(c for c in cases if c.scenario == "known name, redesigned layout" and c.doc_id == EX_DOC and c.page == 1)
g_red = gate.page_rules(redesign.evidence, redesign.vendor_name)
_, shown_red = E.ungated_shown(graph, post, redesign)
fig, ax = plt.subplots(figsize=(7, 9))
viz.page_axes(corpus.pdf(EX_DOC), 1, ax, fade=0.5,
              title=f"Redesign: true boxes (blue) · ungated memory (orange) · P(NEW) = {g_red['p_new']:.2f}")
for f in redesign.gt["fields"].values():
    if f["bbox"]:
        viz.draw_box(ax, f["bbox"], viz.BLUE)
for key, r in shown_red.items():
    viz.draw_range(ax, r["range"], viz.ORANGE, key)
for r in g_red["rules"]:
    if r["gate"] == "box":
        viz.draw_range(ax, r["range"], viz.AQUA, f"gated: {r['name']}", ls="-")
plt.show()

**What to notice.** For unseen vendors and redesigns, the ungated guide puts 5–6 wrong positions in every prompt.
The gate sends almost all of those pages to NEW and shows next to nothing, which is the v2 failure fixed. For a
known layout under a new name, structure partly overrides the name: about half the pages are still placed in the
right tree, and when they are, the boxes are right.

### E5. Sensitivity: where does it break?
The same studies under heavier box noise, and with the field term **tempered** ($T$ > 1 flattens naive Bayes,
whose independent-field assumption overcounts correlated evidence). Read `gate_box`, `field_temperature` and
`box_range_min` against this table.

In [ ]:
sweep = []
for box_miss in (0.0, 0.03, 0.057, 0.1):
    g_cfg = replace(cfg, noise_box_miss=box_miss)
    sweep_gen = ScenarioGenerator(corpus, graph, g_cfg)
    sweep_cases = [c for c in sweep_gen.baseline() + sweep_gen.no_vendor() + sweep_gen.unseen_and_drift()
                   if c.meta["noisy"]]
    for T in (1.0, 2.0, 4.0):
        p_, r_ = E.evaluate(gate.with_posterior(post.with_settings(field_temperature=T)), sweep_cases)
        shown_ = r_[r_.gate == "box"]
        sweep.append({"box_miss": box_miss, "T": T, **E.calibration_scores(r_), "map_acc": p_.correct.mean(),
                      "share_box": (r_.gate == "box").mean(), "box_precision": shown_.hit.mean(),
                      "wrong_per_page": (~shown_.hit).sum() / len(p_)})
sweep = pd.DataFrame(sweep)
display(sweep.round(3))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
for T, color in zip((1.0, 2.0, 4.0), (viz.BLUE, viz.ORANGE, viz.AQUA)):
    s = sweep[sweep["T"] == T]
    a1.plot(s.box_miss, s.map_acc, marker="o", color=color, lw=2, label=f"T = {T:g}")
    a2.plot(s.box_miss, s.brier, marker="o", color=color, lw=2, label=f"T = {T:g}")
a1.set(xlabel="mean box-centre miss", ylabel="MAP accuracy", ylim=(0, 1)); a1.set_title("Identification", loc="left")
a2.set(xlabel="mean box-centre miss", ylabel="Brier score (lower is better)"); a2.set_title("Calibration", loc="left")
a1.axvline(0.057, color=viz.AXIS, ls="--", lw=1); a2.axvline(0.057, color=viz.AXIS, ls="--", lw=1)
a1.legend(); plt.tight_layout(); plt.show()

**What to notice.**
* **The gate fails safe.** As box noise doubles past Gemini's level (0.057 → 0.1), identification drops to ~0.55.
  But the gate shows fewer positions, so wrong positions per page *fall* rather than rise. The cost is coverage,
  not trust.
* **Tempering doesn't help here.** $T$ = 2 or 4 leaves calibration flat at realistic noise and worsens it at high
  noise. On this corpus the field term isn't the source of overconfidence. To tolerate noisier extractions, widen
  `box_range_min` instead.

### Overview: every scenario at once

In [ ]:
viz.gate_mix_plot(summary)
plt.show()
effect

---
## Part IV: Does the graph help the LLM?
`gating.ipynb` runs the real comparison with Gemini: call 0 (unguided), pass 1 (ungated memory) and pass 2 (gated
memory) on the held-out third, saved under `results/`. This cell summarizes those runs when they're present.

In [ ]:
llm = E.llm_fields(corpus)
if llm is None:
    display(Markdown(f"*No saved Gemini passes under `{cfg.results_dir.resolve()}`. Run `gating.ipynb` (or copy its "
                     "`results/` folder here) to fill this section.*"))
else:
    display(llm.groupby("setup").apply(E.summarize, include_groups=False)[E.LLM_METRICS].round(3))
    display(E.llm_table(llm))

**How to read it.**
* **Known vendors (buckets 2+):** memory should raise `center_within_tol` over unguided, and the gate should keep
  that gain.
* **`0 (unseen)`:** the ungated memory lowered box accuracy here in v2. The gate should bring it back to at least
  the unguided level, because E4 shows it withholds almost everything on these pages.
* **Value accuracy** moves less than box accuracy: memory mostly tells the model *where* to look.

---
## Limits and next steps
* **Confident misidentification of twins.** When a misread name points at a structural twin, rules where the twins
  disagree keep high confidence (E2). Remedies: cross-check the name against fields such as phone, tax ID or bank
  numbers, or cap confidence when a twin with a disagreeing range is close in the posterior.
* **Thin trees.** Vendors with one or two invoices are hard to identify without a name (E3). Each new invoice
  strengthens them, so memory should be written back after a verified extraction.
* **Heavier box noise costs coverage.** Beyond ~0.06 page-widths of miss, identification falls quickly (E5). The
  gate stays safe, but it shows less. Ranges should widen with the measured noise of the extractor in use.
* **Generated pages are not Gemini.** The noise channel matches Gemini's average errors, not their structure (for
  example, misses that cluster on scans). Part IV is the real-world check.

## Appendix: the formulas
**Rule reliability.** Range $R = [c_x \pm m] \times [c_y \pm m]$ with $m = \mathrm{clip}(2\,\mathrm{spread}, 0.06, 0.15)$.
With $s$ of $n$ training centres inside, and the peak excluded:
$q = \dfrac{(s-1)+\alpha}{(n-1)+\alpha+\beta}$, where $\mathrm{Beta}(\alpha,\beta)$ is fitted so that its mean is
the out-of-sample hit rate of one-document rules.

**Posterior.** $P(\text{NEW}) = N_1/N$ (Good–Turing) and $P(L) = (1 - P(\text{NEW}))\,n_L/\sum n_{L'}$.
Name: $1-\varepsilon$ for the right key, $\varepsilon\rho$ for a variant spelling, and $\varepsilon(1-\rho)/M$
otherwise. A missing name gives 1. Fields: $\theta_L = (n_{L,k} + m\,\theta_{\text{ALL},k})/(n_L + m)$,
$P(x_k{=}1 \mid L) = \theta(1-\text{miss}) + (1-\theta)\,\text{false\_fill}$, and the product is raised to $1/T$.
Boxes: $q/|R|$ inside, $(1-q)/(1-|R|)$ outside, and 1 where $L$ has no range.

**Confidence.** $\text{conf}_k = \sum_L P(L\mid e)\,h_L$, with $h_{L^*} = q_{L^*}$. For another layout with a
range, $h_L = q_L\frac{|R_L\cap R^*|}{|R_L|} + (1-q_L)\frac{|R^*| - |R_L\cap R^*|}{1-|R_L|}$, and otherwise
$h_L = |R^*|$.

In [ ]:
graph.close()